In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
dates = pd.date_range(start="2025-01-01", periods=100, freq='D')
prices = 100 + np.random.randn(100).cumsum()
df = pd.DataFrame(data={'Price': prices}, index=dates)

df['Short_MA'] = df['Price'].rolling(window=5).mean()
df['Long_MA'] = df['Price'].rolling(window=20).mean()

df['Signal'] = np.where(df['Short_MA'] > df['Long_MA'], 1, 0)
df['Position'] = df['Signal'].diff()

df['Market_Returns'] = df['Price'].pct_change()
df['Strategy_Returns'] = df['Market_Returns'] * df['Signal'].shift(1)


total_market_returns = (df['Market_Returns'] + 1).prod() - 1
total_strategy_returns = (df['Strategy_Returns'] + 1).prod() - 1

print(f"Total Market Returns: {total_market_returns:.2%}")
print(f"Total Strategy Returns: {total_strategy_returns:.2%}")


Total Market Returns: -10.83%
Total Strategy Returns: -5.73%


In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf

ticker = "AAPL"
df = yf.download(ticker, start="2024-01-01", end="2026-01-01")

df = df[['Close']].copy()
df.columns = ['Price']

df['Short_MA'] = df['Price'].rolling(window=10).mean()
df['Long_MA'] = df['Price'].rolling(window=50).mean()

df['Signal'] = np.where(df['Short_MA'] > df['Long_MA'], 1, 0)
df['Position'] = df['Signal'].diff()

df['Market_Returns'] = df['Price'].pct_change()
df['Strategy_Returns'] = df['Market_Returns'] * df['Signal'].shift(1)

fee_rate = 0.001
df['Transaction_Costs'] = df['Position'].abs() * fee_rate
df['Net_Strategy_Returns'] = df['Strategy_Returns'] - df['Transaction_Costs']

df.dropna(inplace=True)

total_market_returns = (df['Market_Returns'] + 1).prod() - 1
total_strategy_returns = (df['Strategy_Returns'] + 1).prod() - 1

print(f"---Backtest Results for {ticker}---")
print(f"Buy-and-Hold Market Return: {total_market_returns:.2%}")
print(f"Moving Average Strategy Return (Net of Fees): {total_strategy_returns:.2%}")





/tmp/ipykernel_2844/3574070701.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start="2024-01-01", end="2026-01-01")
[*********************100%***********************]  1 of 1 completed

---Backtest Results for AAPL---
Buy-and-Hold Market Return: 58.22%
Moving Average Strategy Return (Net of Fees): 33.77%
